# AstroHunter KZ Phase 1: Beta Pic Positive Control

This notebook is a compact technical proof-of-concept for the Phase 1 pipeline. It downloads one public TESS light curve for Beta Pic / TIC 270577175, cleans and normalizes the data, detects simple candidate dip-like features, computes simple asymmetry scores, and saves reproducible figures and a candidate table.

Scientific caution: this repository identifies candidate events only. It does not claim confirmed exocomet discovery. The strongest Phase 1 features may be instrumental/systematic and require quality-flag and multi-sector validation.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import matplotlib.pyplot as plt

from astrohunter.asymmetry import add_asymmetry_scores, detect_candidate_dips
from astrohunter.lightcurves import (
    clean_normalize_lightcurve,
    download_limited_lightcurves,
    lightcurve_to_dataframe,
)
from astrohunter.plotting import (
    plot_event_window,
    plot_full_lightcurve,
    plot_lightcurve_with_events,
    plot_zoom_window,
)

TARGET = "TIC 270577175"
MAX_LIGHTCURVES = 1
SIGMA_THRESHOLD = 4.0
WINDOW_DAYS = 0.5

FIGURES_DIR = REPO_ROOT / "results" / "figures"
TABLES_DIR = REPO_ROOT / "results" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## Download One Public TESS Light Curve

The quickstart intentionally limits downloads to one light curve by default so the notebook remains lightweight and reproducible.

In [ ]:
collection = download_limited_lightcurves(TARGET, mission="TESS", max_lightcurves=MAX_LIGHTCURVES)
lc = collection[0] if len(collection) == 1 else collection.stitch()
lc = clean_normalize_lightcurve(lc)
df = lightcurve_to_dataframe(lc)

time = df["time_btjd"].to_numpy()
flux = df["flux"].to_numpy()

df.head(), len(df)

## Plot the Full Light Curve

In [ ]:
fig = plot_full_lightcurve(
    time,
    flux,
    "Beta Pic / TIC 270577175 TESS Light Curve",
    output_path=FIGURES_DIR / "beta_pic_full_lightcurve.png",
)
fig

## Detect Candidate Dip-Like Features

In [ ]:
events = detect_candidate_dips(
    time,
    flux,
    sigma_threshold=SIGMA_THRESHOLD,
    min_distance=5,
    window_days=WINDOW_DAYS,
)
events = add_asymmetry_scores(events, time, flux, window_days=WINDOW_DAYS)

table_path = TABLES_DIR / "beta_pic_candidate_dips.csv"
events.to_csv(table_path, index=False)
events.head(10)

## Plot Candidate Events

In [ ]:
fig = plot_lightcurve_with_events(
    time,
    flux,
    events,
    "Beta Pic TESS Light Curve with Candidate Dip-Like Features",
    output_path=FIGURES_DIR / "beta_pic_lightcurve_with_detected_dips.png",
)
fig

In [ ]:
if not events.empty:
    strongest = events.sort_values("depth", ascending=False).iloc[0]
    start = strongest["event_time_btjd"] - WINDOW_DAYS / 2
    end = strongest["event_time_btjd"] + WINDOW_DAYS / 2
    fig = plot_zoom_window(
        time,
        flux,
        start,
        end,
        f"Beta Pic Strongest Candidate Dip-Like Feature near BTJD {strongest['event_time_btjd']:.3f}",
        output_path=FIGURES_DIR / "beta_pic_zoom_strongest_dip.png",
    )
else:
    fig = None
fig

In [ ]:
for idx, row in events.head(3).iterrows():
    fig = plot_event_window(
        time,
        flux,
        row["event_time_btjd"],
        window_days=WINDOW_DAYS,
        title=f"Candidate Dip-Like Feature {idx + 1} near BTJD {row['event_time_btjd']:.3f}",
        output_path=FIGURES_DIR / f"beta_pic_candidate_window_{idx + 1:02d}.png",
    )
    plt.show()

## Phase 1 Interpretation Boundary

This notebook is a technical positive control only. Candidate dip-like features are not confirmed exocomets. Follow-up work must add quality-flag vetting, multi-sector consistency checks, catalog controls, injection recovery, and statistical comparison against matched non-disk stars.